# Étape 3 — Nettoyage et dédoublonnage exact

Objectif : visualiser chaque transformation, mesurer les suppressions et comparer le résultat au traitement réutilisable de `00_fonctions_base.ipynb`.

Toutes les opérations sont effectuées en mémoire : la base et ses textes originaux restent inchangés. Ce notebook n'ajoute aucun filtrage d'actualité.

## 1. Chargement des données

In [1]:
from pathlib import Path
import sys

# Fonctionne depuis le notebook, le sous-projet ou la racine du projet.
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir le notebook depuis le projet Bluesky ou son dossier notebooks.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATABASE_PATH = ROOT / "posts.db"
print("Sous-projet :", ROOT.name)
print("Base SQLite :", (Path(ROOT.name) / DATABASE_PATH.relative_to(ROOT)).as_posix())

Sous-projet : bluesky-news-mining
Base SQLite : bluesky-news-mining/posts.db


In [2]:
import pandas as pd
from IPython.display import display

import json

# Charger les fonctions communes, sans lire ni modifier la base.
base_path = ROOT / "notebooks" / "00_fonctions_base.ipynb"
base_notebook = json.loads(base_path.read_text(encoding="utf-8"))
for definition_cell in base_notebook["cells"]:
    if definition_cell["cell_type"] == "code" and "database-definitions" in definition_cell["metadata"].get("tags", []):
        exec(compile("".join(definition_cell["source"]), str(base_path), "exec"), globals())

pd.set_option("display.max_colwidth", 120)
posts_raw = load_posts(DATABASE_PATH, limit=None)
print("Publications chargées :", len(posts_raw))

Publications chargées : 193


## 2. Vérification de la langue déclarée

On conserve les posts dont `langs` contient exactement `fr`. Il s'agit du même contrôle que lors de la collecte ; un post `en,fr` est accepté.

In [3]:
is_french = posts_raw["langs"].fillna("").str.split(",").map(lambda langs: "fr" in langs).astype(bool)
posts_fr = posts_raw.loc[is_french].copy()
print("Publications déclarées en français :", len(posts_fr))

Publications déclarées en français : 193


## 3. Nettoyage par expressions régulières

La fonction `clean_text` supprime les URL commençant par `http://`, `https://` ou `www.`, normalise les espaces et retire les espaces aux extrémités. La casse, la ponctuation, les accents et les emojis sont conservés. Il n'y a ni lemmatisation ni suppression des mots vides.

La colonne `text` conserve l'original ; `clean_text` reçoit la version nettoyée.

In [4]:
example = "  Grève demain !\nhttps://exemple.fr/article\t Rendez-vous à Lyon.  "
print("Avant :", repr(example))
print("Après :", repr(clean_text(example)))

posts_cleaned = posts_fr.copy()
posts_cleaned["clean_text"] = posts_cleaned["text"].map(clean_text)
display(posts_cleaned[["text", "clean_text"]].head(10))

Avant : '  Grève demain !\nhttps://exemple.fr/article\t Rendez-vous à Lyon.  '
Après : 'Grève demain ! Rendez-vous à Lyon.'


,text,clean_text
0,,
1,,
2,,
3,,
4,,
5,,
6,#dani daniels,#dani daniels
7,bsky.app/profile/brev...,bsky.app/profile/brev...
8,Pzs pour se dire que faudrait ptet une piece plus grande?avoir une version avec un candidat moins stressé?\n\nYen a ...,Pzs pour se dire que faudrait ptet une piece plus grande?avoir une version avec un candidat moins stressé? Yen a pas...
9,Alors ok il est à l'aise comme séminariste qui se serait fait surprendre par le curé en train de se palucher\nMais l...,Alors ok il est à l'aise comme séminariste qui se serait fait surprendre par le curé en train de se palucher Mais l'...


## 4. Retrait des textes trop courts

Après suppression des URL, un post peut être vide ou ne plus contenir que quelques caractères. Le seuil actuel est de 3 caractères minimum ; il ne s'agit pas d'un nombre de mots.

In [5]:
long_enough = posts_cleaned["clean_text"].str.len() >= MIN_TEXT_LENGTH
posts_long = posts_cleaned.loc[long_enough].copy()
print("Longueur minimale :", MIN_TEXT_LENGTH, "caractères")
print("Publications retirées pour longueur insuffisante :", len(posts_cleaned) - len(posts_long))

Longueur minimale : 3 caractères
Publications retirées pour longueur insuffisante : 20


## 5. Dédoublonnage exact des textes nettoyés

Deux lignes sont considérées comme doublons si leur `clean_text` est strictement identique. La comparaison est sensible à la casse et à la ponctuation. Deux textes ayant des URL différentes peuvent devenir identiques après nettoyage.

On conserve la première occurrence chargée, donc la plus récente selon la date utilisée au chargement (avec l'URI comme second critère de tri). Les doublons restent dans SQLite : seule la table en mémoire est réduite.

In [6]:
duplicate_mask = posts_long["clean_text"].duplicated(keep="first")
print("Occurrences dupliquées retirées :", int(duplicate_mask.sum()))
display(posts_long.loc[posts_long["clean_text"].duplicated(keep=False), ["uri", "text", "clean_text"]].head(10))

posts_unique = posts_long.drop_duplicates(subset="clean_text", keep="first").reset_index(drop=True)
print("Publications nettoyées et uniques :", len(posts_unique))

Occurrences dupliquées retirées : 0
Publications nettoyées et uniques : 173


,uri,text,clean_text


## 6. Bilan des étapes

In [7]:
counts = [
    ("Chargement", len(posts_raw)),
    ("Langue déclarée fr", len(posts_fr)),
    ("Nettoyage des URL et espaces", len(posts_cleaned)),
    ("Longueur minimale", len(posts_long)),
    ("Dédoublonnage exact", len(posts_unique)),
]
summary = pd.DataFrame(counts, columns=["Étape", "Publications"])
summary["Retirées à cette étape"] = (summary["Publications"].shift(1) - summary["Publications"]).fillna(0).astype(int)
display(summary)
display(posts_unique[["created_at", "did", "text", "clean_text"]].head(10))

,Étape,Publications,Retirées à cette étape
0,Chargement,193,0
1,Langue déclarée fr,193,0
2,Nettoyage des URL et espaces,193,0
3,Longueur minimale,173,20
4,Dédoublonnage exact,173,0


,created_at,did,text,clean_text
0,2026-11-08 10:54:39.033000+00:00,did:plc:zezjihophg7t5b33einwykm2,#dani daniels,#dani daniels
1,2026-11-08 09:17:48.649000+00:00,did:plc:rccl5vxskabllxbnyqloyyih,bsky.app/profile/brev...,bsky.app/profile/brev...
2,2026-11-08 09:17:15.140000+00:00,did:plc:xqfzs4wfwchkj5lv7xhmh53q,Pzs pour se dire que faudrait ptet une piece plus grande?avoir une version avec un candidat moins stressé?\n\nYen a ...,Pzs pour se dire que faudrait ptet une piece plus grande?avoir une version avec un candidat moins stressé? Yen a pas...
3,2026-11-08 09:17:15.139000+00:00,did:plc:xqfzs4wfwchkj5lv7xhmh53q,Alors ok il est à l'aise comme séminariste qui se serait fait surprendre par le curé en train de se palucher\nMais l...,Alors ok il est à l'aise comme séminariste qui se serait fait surprendre par le curé en train de se palucher Mais l'...
4,2026-11-08 09:17:14.307000+00:00,did:plc:a3taglznovt2jugkgzor4e5b,"Trop obscure, elle gagne à être connue.","Trop obscure, elle gagne à être connue."
5,2026-11-08 09:17:14.000000+00:00,did:plc:6g5uqiiv2j75kdumhjkujvrk,"La plante des abeilles 🌿\nMélissa, en grec, c'est l'abeille.\nDisponible à Paris chez Herba Barona 👉 paris-herbabaro...","La plante des abeilles 🌿 Mélissa, en grec, c'est l'abeille. Disponible à Paris chez Herba Barona 👉 paris-herbabarona..."
6,2026-11-08 09:17:13.453000+00:00,did:plc:ktzgtmlf3j5oozegh7b4qs63,PROUVE-LE,PROUVE-LE
7,2026-11-08 09:17:12.291000+00:00,did:plc:komdqh75tlba6mwsbyviiyzi,"👍 Ahah, c'est pas faux ! \nTrès bon dimanche Jonathan ! 💫","👍 Ahah, c'est pas faux ! Très bon dimanche Jonathan ! 💫"
8,2026-11-08 09:17:11.470000+00:00,did:plc:4lh5urmx2ncgwf7ka7v4hd5v,Morning Nice sunday to U too,Morning Nice sunday to U too
9,2026-11-08 09:17:11.206127+00:00,did:plc:buuvp6swhof5ymyxfpt3qtbn,Jazz live – Step 1\nLe 08/12/2026\n\nLes étudiants en jazz vous propose un concert haut en couleur !\nhttps://www.to...,Jazz live – Step 1 Le 08/12/2026 Les étudiants en jazz vous propose un concert haut en couleur ! #VilleDeTours #Tours


## 7. Vérifier le résultat

Le traitement détaillé doit donner exactement le même résultat que `preprocess_posts`. On vérifie aussi l'absence de doublons et le seuil de longueur.

In [8]:
expected = preprocess_posts(posts_raw)
pd.testing.assert_frame_equal(posts_unique, expected)
assert posts_unique["clean_text"].is_unique
assert (posts_unique["clean_text"].str.len() >= MIN_TEXT_LENGTH).all()
assert len(load_posts(DATABASE_PATH, limit=None)) == len(posts_raw), "Le nombre de lignes de la base a changé pendant le notebook."
print("Résultat identique à preprocess_posts : OK.")
print("Absence de doublons et longueur minimale : OK.")
print("Publications brutes :", len(posts_raw), "| conservées :", len(posts_unique))

Résultat identique à preprocess_posts : OK.
Absence de doublons et longueur minimale : OK.
Publications brutes : 193 | conservées : 173
